# Employee promotion prediction tool

## Import the dependencies

In [225]:
import torch
import torch.nn as nn
import torch.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
url = 'https://raw.githubusercontent.com/MainakRepositor/Datasets/refs/heads/master/HR%20Data/train.csv'

## Create a dataset

In [226]:
class EmployeeDataSet(Dataset):
    def __init__(self, url, transform=None):
        super(EmployeeDataSet, self).__init__()
        # load the dataset
        self.employee_data = pd.read_csv(url)
        self.transform = transform

        # clean the dataset | remove unnecessary columns
        self.employee_data = pd.get_dummies(self.employee_data)
        self.X_labels = self.employee_data.drop(columns=['is_promoted', 'employee_id'])
        self.y_target = self.employee_data['is_promoted']

        # scale the X label
        self.scaler = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(self.X_labels)
        # transform X and y into tensors for pytorch
        self.X_tensor = torch.tensor(self.X_scaled, dtype=torch.float32)
        self.y_tensor = torch.tensor(self.y_target.astype(float).values, dtype=torch.float32)

    def __len__(self):
        # return size of the dataset
        return len(self.X_labels)
    def __getitem__(self, idx):
        # return idx tensor
        return self.X_tensor[idx], self.y_tensor[idx]

## Prepare data for loading

### Create dataset

In [227]:
dataset = EmployeeDataSet(url)

### Split the dataset into test and training

In [228]:
total_size = len(dataset)
train_size = int(total_size * 0.7)
test_size = total_size - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

### Hook the training and test sets into their loaders

In [229]:
# weights = [class_weights[int(label)] for label in dataset.y_target]
# sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
# sampler=sampler
training_loader = DataLoader(dataset=train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(dataset=test_dataset,batch_size=32,shuffle=False)

## Create model architecture

In [230]:
class HrPromotionTool(nn.Module):
    def __init__(self, in_features_=58, h1=64, h2=32):
        super(HrPromotionTool, self).__init__()
        self.IN = nn.Linear(in_features_, h1)
        self.fc1 = nn.Linear(h1, h2)
        self.OUT = nn.Linear(h2, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # let the data pass through the neurons
        x = torch.relu(self.IN(x))
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.OUT(x)
        return x

## Training loop

### The preparations

In [231]:
model = HrPromotionTool()
pos_weight = torch.tensor([50.0])
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # where lr is learning rate how precise is each step of descend
        #
        # correct_count = predicted_classes.eq(y_batch.unsqueeze(1)).sum().item()
        # correct_percent = correct_count / y_batch.size(0) * 100
        # print(f'Model got {correct_count} correct out of {y_batch.size(0)} samples with percentage {correct_percent}%')

### The loop itself

In [232]:
epochs = 5
for epoch in range(epochs):
    running_tp = 0
    running_fp = 0
    running_actually_promoted = 0
    running_predicted_positives = 0

    for X_batch, y_batch in training_loader:
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch.unsqueeze(1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predicted_classes = (predictions > 0).float() # (if sth) ?'then' do this :'or' this
        tar = y_batch.unsqueeze(1).float()
        batch_tp = (predicted_classes * tar).sum().item()
        batch_actually_promoted = tar.sum().item()
        running_predicted_positives += predicted_classes.sum().item()
        print(running_predicted_positives)

        running_tp += batch_tp
        running_actually_promoted += batch_actually_promoted
    epoch_precision = running_tp / (running_predicted_positives + 1e-7)
    epoch_recall = running_tp / (running_actually_promoted + 1e-7)
    print(f'Epoch: {epoch + 1} Recall: {epoch_recall:.6f}')



3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0
3.0


## Test loop

In [233]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch.unsqueeze(1))
        total += y_batch.size(0)
        predicted_classes = (predictions > 0).float()
        correct += predicted_classes.eq(y_batch.unsqueeze(1)).sum().item()
    print(f'Model has scored {correct} correct out of {total} samples with percentage of correct predictions {100 * correct / total:.2f}%')

Model has scored 15040 correct out of 16443 samples with percentage of correct predictions 91.47%
